# QTRAP Parse and Plot

# 1 - QTRAP Parse

# call python file

In [ ]:
import re
import os
import pandas as pd

def TIC_RSD(intensities):
    """
    Calculate the Relative Standard Deviation (RSD) for a given intensity array.

    Args:
        intensities (list of int): The intensity values.

    Returns:
        float: The RSD percentage.
    """
    if not intensities:
        return 0.0
    mean = sum(intensities) / len(intensities)
    if mean == 0:
        return 0.0
    variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
    std_dev = variance ** 0.5
    rsd = (std_dev / mean) * 100
    return rsd

def parse_chromatogram_data(input_dir, output_dir):
    """
    Parse chromatogram data from text files in the specified input directory
    and save the results as a CSV in the specified output directory.

    Args:
        input_dir (str): Directory containing the input text files.
        output_dir (str): Directory to save the output CSV file.

    Returns:
        pd.DataFrame: DataFrame containing the parsed chromatogram data.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'parsed_chromatogram_data.csv')

    # Initialize lists to store parsed data
    filenames = []
    q1_values = []
    q3_values = []
    lipids = []
    dates = []          # List for Date
    sample_names = []   # List for Sample_Name
    samples = []        # List for Sample
    summed_intensities = []  # List for Summed_Intensity
    tic_rsd_values = [] # List for TIC_RSD

    # Iterate through all .txt files in the specified directory
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(input_dir, file_name)
            print(f"\nProcessing file: {file_name}")  # Debug: Current file being processed

            # Extract Date and Sample_Name from the filename
            base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
            parts = base_name.split('_', 1)             # Split only on the first underscore
            if len(parts) == 2:
                date_str = parts[0]                     # e.g., '20241115'
                sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
            else:
                # Handle unexpected filename formats
                date_str = ''
                sample_name = ''
                print(f"Warning: Unexpected filename format for file {file_name}")

            # Determine the Sample value based on Sample_Name
            sample = 'Blank' if 'Blank' in sample_name else 'Sample'

            # Open and read all lines of the file
            with open(file_path, 'r') as file:
                lines = file.readlines()

            # -------------------
            # Extract TIC Data
            # -------------------
            TIC_intensities = []
            for i, line in enumerate(lines):
                if 'id: TIC' in line:
                    print("Found 'id: TIC' section.")  # Debug: Found TIC section
                    # Look for the intensity array within TIC section
                    for j in range(i, len(lines)):
                        if 'binaryDataArray:' in lines[j] and 'intensity array' in lines[j]:
                            print("Found 'cvParam: intensity array, number of detector counts'.")  # Debug
                            # The intensity values are in the next line
                            if j + 1 < len(lines):
                                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', lines[j + 1])
                                if match:
                                    TIC_intensities = list(map(int, match.group(1).split()))
                                    print(f"Extracted TIC intensities: {TIC_intensities}")  # Debug: TIC values
                                    break
                            else:
                                print(f"Warning: No intensity data found after line {j} in file {file_name}")
                    break  # Assuming only one TIC section per file

            if not TIC_intensities:
                print(f"Warning: No TIC intensities found in file {file_name}")
            else:
                # Calculate TIC_RSD using the TIC_RSD function
                TIC_rsd = TIC_RSD(TIC_intensities)
                print(f"Calculated TIC_RSD: {TIC_rsd:.2f}%")  # Debug: TIC RSD value
            # -------------------

            # -------------------
            # Parse Lipid Data
            # -------------------
            current_filename = ""
            current_q1 = None
            current_q3 = None
            current_lipid = ""
            parsing_intensity = False
            intensities = []

            for line in lines:
                # Extract the filename (assumes filename appears earlier in the file)
                if 'sourceFile:' in line or 'name:' in line:
                    match = re.search(r'name:\s+([\w.]+)', line)
                    if match:
                        current_filename = match.group(1)
                        # Debug: Extracted internal filename
                        print(f"Extracted internal filename: {current_filename}")

                # Extract Q1, Q3 values, and Lipid name
                if 'id: SRM SIC Q1=' in line:
                    match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                    if match:
                        current_q1 = float(match.group(1))
                        current_q3 = float(match.group(2))
                        current_lipid = match.group(3)
                        # Debug: Extracted Q1, Q3, and Lipid
                        print(f"Extracted Q1: {current_q1}, Q3: {current_q3}, Lipid: {current_lipid}")

                # Check if we are parsing intensity array data for lipids
                if 'cvParam: intensity array' in line and 'number of detector counts' not in line:
                    parsing_intensity = True
                    intensities = []
                    print("Parsing lipid intensity array.")  # Debug
                elif parsing_intensity and 'binary: [' in line:
                    # Extract intensity values
                    match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                    if match:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                        print(f"Extracted lipid intensities: {intensities}")  # Debug: Lipid intensity values
                        print(f"Summed Intensity: {current_intensity_sum}")  # Debug: Summed intensity

                        # Append the extracted and calculated data to the lists
                        if current_filename and current_q1 is not None and current_q3 is not None:
                            filenames.append(current_filename)
                            q1_values.append(current_q1)
                            q3_values.append(current_q3)
                            lipids.append(current_lipid)
                            dates.append(date_str)              # Append Date
                            sample_names.append(sample_name)    # Append Sample_Name
                            samples.append(sample)              # Append Sample
                            summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                            tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                            print(f"Appended data for lipid: {current_lipid}")  # Debug: Data appended

                    parsing_intensity = False

    # Create a DataFrame
    chromatogram_df = pd.DataFrame({
        'Date': dates,                        # Date column
        'Sample_Name': sample_names,          # Sample_Name column
        'Sample': samples,                    # Sample column
        'Lipid': lipids,
        'Q1': q1_values,
        'Q3': q3_values,
        'Summed_Intensity': summed_intensities,  # Summed_Intensity column
        'TIC_RSD': tic_rsd_values,            # TIC_RSD column
        'Filename': filenames,
    })

    # Create Base_Sample_Name by removing 'Blank_' prefix if present
    chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

    # Assign group numbers based on Base_Sample_Name
    chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

    # Optionally, drop the Base_Sample_Name column if not needed
    chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

    # Optionally, convert Date to datetime format
    # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

    # Reorder columns for better readability
    columns_order = [
        'Date', 
        'Sample_Name', 
        'Sample', 
        'Blank_Group', 
        'Lipid', 
        'Q1', 
        'Q3', 
        'Summed_Intensity',
        'TIC_RSD',                           # Include TIC_RSD in the order
        'Filename'
    ]
    chromatogram_df = chromatogram_df[columns_order]

    # Save the DataFrame to a CSV file
    chromatogram_df.to_csv(output_file, index=False)
    print(f"\nParsing complete. Data saved to {output_file}")  # Debug: Completion message

    return chromatogram_df


: 

In [11]:
import re
import os
import pandas as pd

def TIC_RSD(intensities):
    """
    Calculate the Relative Standard Deviation (RSD) for a given intensity array.

    Args:
        intensities (list of int): The intensity values.

    Returns:
        float: The RSD percentage.
    """
    if not intensities:
        return 0.0
    mean = sum(intensities) / len(intensities)
    if mean == 0:
        return 0.0
    variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
    std_dev = variance ** 0.5
    rsd = (std_dev / mean) * 100
    return rsd

def parse_chromatogram_data(input_dir, output_dir):
    """
    Parse chromatogram data from text files in the specified input directory
    and save the results as a CSV in the specified output directory.

    Args:
        input_dir (str): Directory containing the input text files.
        output_dir (str): Directory to save the output CSV file.

    Returns:
        pd.DataFrame: DataFrame containing the parsed chromatogram data.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'parsed_chromatogram_data.csv')

    # Initialize lists to store parsed data
    filenames = []
    q1_values = []
    q3_values = []
    lipids = []
    dates = []          # List for Date
    sample_names = []   # List for Sample_Name
    samples = []        # List for Sample
    summed_intensities = []  # List for Summed_Intensity
    tic_rsd_values = [] # List for TIC_RSD

    # Iterate through all .txt files in the specified directory
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(input_dir, file_name)
            print(f"\nProcessing file: {file_name}")  # Debug: Current file being processed

            # Extract Date and Sample_Name from the filename
            base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
            parts = base_name.split('_', 1)             # Split only on the first underscore
            if len(parts) == 2:
                date_str = parts[0]                     # e.g., '20241115'
                sample_name = parts[1]                  # e.g., 'Plasma_TG_22-6'
            else:
                # Handle unexpected filename formats
                date_str = ''
                sample_name = ''
                print(f"Warning: Unexpected filename format for file {file_name}")

            # Ensure sample_name is a string
            if not isinstance(sample_name, str):
                sample_name = str(sample_name)
                print(f"Converted Sample_Name to string: {sample_name}")

            # Determine the Sample value based on Sample_Name
            sample = 'Blank' if 'Blank' in sample_name else 'Sample'

            # Open and read all lines of the file
            with open(file_path, 'r') as file:
                lines = file.readlines()

            # -------------------
            # Extract TIC Data
            # -------------------
            TIC_intensities = []
            in_TIC_section = False
            for i, line in enumerate(lines):
                if 'id: TIC' in line:
                    print("Found 'id: TIC' section.")  # Debug: Found TIC section
                    in_TIC_section = True
                    continue  # Move to next line

                if in_TIC_section:
                    # Look for 'cvParam: intensity array, number of detector counts'
                    if 'cvParam: intensity array, number of detector counts' in line:
                        print("Found 'cvParam: intensity array, number of detector counts'.")  # Debug
                        # The intensity values are in the next line
                        if i + 1 < len(lines):
                            match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', lines[i + 1])
                            if match:
                                TIC_intensities = list(map(int, match.group(1).split()))
                                print(f"Extracted TIC intensities: {TIC_intensities}")  # Debug: TIC values
                                break
                            else:
                                print(f"Warning: No intensity data found after line {i + 1} in file {file_name}")
                        else:
                            print(f"Warning: No lines after 'cvParam: intensity array' in file {file_name}")
                    # If another 'id: ' section starts, stop searching
                    elif re.match(r'id:\s+\w+', line):
                        print("Reached a new 'id:' section before finding TIC intensities.")  # Debug
                        break

            if not TIC_intensities:
                print(f"Warning: No TIC intensities found in file {file_name}")
                TIC_rsd = 0.0  # Default value when TIC intensities are not found
            else:
                # Calculate TIC_RSD using the TIC_RSD function
                TIC_rsd = TIC_RSD(TIC_intensities)
                print(f"Calculated TIC_RSD: {TIC_rsd:.2f}%")  # Debug: TIC RSD value

            # -------------------
            # Parse Lipid Data
            # -------------------
            current_filename = ""
            current_q1 = None
            current_q3 = None
            current_lipid = ""
            parsing_intensity = False
            intensities = []

            for line in lines:
                # Extract the filename (assumes filename appears earlier in the file)
                if 'sourceFile:' in line or 'name:' in line:
                    match = re.search(r'name:\s+([\w.]+)', line)
                    if match:
                        current_filename = match.group(1)
                        # Debug: Extracted internal filename
                        print(f"Extracted internal filename: {current_filename}")

                # Extract Q1, Q3 values, and Lipid name
                if 'id: SRM SIC Q1=' in line:
                    match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                    if match:
                        current_q1 = float(match.group(1))
                        current_q3 = float(match.group(2))
                        current_lipid = match.group(3)
                        # Debug: Extracted Q1, Q3, and Lipid
                        # Removed print statement for Q1 extraction as per user request

                # Check if we are parsing intensity array data for lipids
                if 'cvParam: intensity array' in line and 'number of detector counts' not in line:
                    parsing_intensity = True
                    intensities = []
                    print("Parsing lipid intensity array.")  # Debug
                elif parsing_intensity and 'binary: [' in line:
                    # Extract intensity values
                    match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                    if match:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                        print(f"Extracted lipid intensities: {intensities}")  # Debug: Lipid intensity values
                        print(f"Summed Intensity: {current_intensity_sum}")  # Debug: Summed intensity

                        # Append the extracted and calculated data to the lists
                        if current_filename and current_q1 is not None and current_q3 is not None:
                            filenames.append(current_filename)
                            q1_values.append(current_q1)
                            q3_values.append(current_q3)
                            lipids.append(current_lipid)
                            dates.append(date_str)              # Append Date
                            sample_names.append(sample_name)    # Append Sample_Name
                            samples.append(sample)              # Append Sample
                            summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                            tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                            print(f"Appended data for lipid: {current_lipid}")  # Debug: Data appended

                    parsing_intensity = False

    # Create a DataFrame
    chromatogram_df = pd.DataFrame({
        'Date': dates,                        # Date column
        'Sample_Name': sample_names,          # Sample_Name column
        'Sample': samples,                    # Sample column
        'Lipid': lipids,
        'Q1': q1_values,
        'Q3': q3_values,
        'Summed_Intensity': summed_intensities,  # Summed_Intensity column
        'TIC_RSD': tic_rsd_values,            # TIC_RSD column
        'Filename': filenames,
    })

    # Ensure that 'Sample_Name' contains only strings
    chromatogram_df['Sample_Name'] = chromatogram_df['Sample_Name'].astype(str)

    # Create Base_Sample_Name by removing 'Blank_' prefix if present
    chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

    # Assign group numbers based on Base_Sample_Name
    chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

    # Optionally, drop the Base_Sample_Name column if not needed
    chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

    # Optionally, convert Date to datetime format
    # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

    # Reorder columns for better readability
    columns_order = [
        'Date', 
        'Sample_Name', 
        'Sample', 
        'Blank_Group', 
        'Lipid', 
        'Q1', 
        'Q3', 
        'Summed_Intensity',
        'TIC_RSD',                           # Include TIC_RSD in the order
        'Filename'
    ]
    chromatogram_df = chromatogram_df[columns_order]

    # Check if all columns have the same length
    lengths = [len(chromatogram_df[col]) for col in chromatogram_df.columns]
    if len(set(lengths)) != 1:
        print("Error: Mismatched list lengths detected. Please check the parsing logic.")
    else:
        print(f"\nAll columns have consistent lengths: {lengths[0]} entries each.")

    # Save the DataFrame to a CSV file
    chromatogram_df.to_csv(output_file, index=False)
    print(f"\nParsing complete. Data saved to {output_file}")  # Debug: Completion message

    return chromatogram_df

# Example usage:
# input_directory = '/path/to/input_directory'
# output_directory = '/path/to/output_directory'
# chromatogram_df = parse_chromatogram_data(input_directory, output_directory)
# print(chromatogram_df.head())


In [12]:
# # Import the function from QTRAP.py
# from QTRAP import parse_chromatogram_data

# Define input and output directories
input_dir = 'data/text/'
output_dir = 'result/'

# Parse the chromatogram data and save to a CSV
chromatogram_df = parse_chromatogram_data(input_dir, output_dir)

# Display the resulting DataFrame
chromatogram_df



Processing file: 20241115_Plasma_TG_22-6.txt
Found 'id: TIC' section.
Found 'cvParam: intensity array, number of detector counts'.
Extracted TIC intensities: [2040, 1440, 1720, 1720, 1760, 2560, 2000, 2520, 1600, 2360, 2000, 1920, 1920, 2200, 1920, 2120, 2400, 2760, 1840, 1800, 2400, 2320, 2080, 1840, 1960, 1760, 1960, 2520, 2200, 1360, 2200, 2720, 1880, 1880, 2080, 2400, 1800, 1720, 1800, 2120, 1880, 2040, 2080, 2760, 2040, 2280, 1840, 2520, 2720, 2040, 2280, 2120, 1760, 2120, 1720, 2280, 1840, 1720, 1400, 2400]
Calculated TIC_RSD: 16.03%
Extracted internal filename: 20241115_Plasma_TG_22
Extracted internal filename: 20241115_Plasma_TG_22

Processing file: 20241115_Plasma_Blank_TG_22-6.txt
Found 'id: TIC' section.
Found 'cvParam: intensity array, number of detector counts'.
Extracted TIC intensities: [1880, 1920, 1160, 1240, 1960, 1520, 1320, 1920, 1560, 2000, 1720, 1760, 1400, 1880, 1960, 1720, 2200, 1760, 1640, 1760, 1880, 1920, 1960, 1760, 1680, 1640, 1640, 1160, 1400, 1560, 1480,

,Date,Sample_Name,Sample,Blank_Group,Lipid,Q1,Q3,Summed_Intensity,TIC_RSD,Filename


In [16]:
import re
import os
import pandas as pd

def TIC_RSD(intensities):
    """
    Calculate the Relative Standard Deviation (RSD) for a given intensity array.

    Args:
        intensities (list of int): The intensity values.

    Returns:
        float: The RSD percentage.
    """
    if not intensities:
        return 0.0
    mean = sum(intensities) / len(intensities)
    if mean == 0:
        return 0.0
    variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
    std_dev = variance ** 0.5
    rsd = (std_dev / mean) * 100
    return rsd

def parse_chromatogram_data(input_dir, output_dir):
    """
    Parse chromatogram data from text files in the specified input directory
    and save the results as a CSV in the specified output directory.

    Args:
        input_dir (str): Directory containing the input text files.
        output_dir (str): Directory to save the output CSV file.

    Returns:
        pd.DataFrame: DataFrame containing the parsed chromatogram data.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'parsed_chromatogram_data.csv')

    # Initialize lists to store parsed data
    filenames = []
    q1_values = []
    q3_values = []
    lipids = []
    dates = []          # List for Date
    sample_names = []   # List for Sample_Name
    samples = []        # List for Sample
    summed_intensities = []  # List for Summed_Intensity
    tic_rsd_values = [] # List for TIC_RSD

    # Iterate through all .txt files in the specified directory
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(input_dir, file_name)
            print(f"\nProcessing file: {file_name}")  # Debug: Current file being processed

            # Extract Date and Sample_Name from the filename
            base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
            parts = base_name.split('_', 1)             # Split only on the first underscore
            if len(parts) == 2:
                date_str = parts[0]                     # e.g., '20241115'
                sample_name = parts[1]                  # e.g., 'Plasma_TG_22-6'
            else:
                # Handle unexpected filename formats
                date_str = ''
                sample_name = ''
                print(f"Warning: Unexpected filename format for file {file_name}")

            # Ensure sample_name is a string
            if not isinstance(sample_name, str):
                sample_name = str(sample_name)
                print(f"Converted Sample_Name to string: {sample_name}")

            # Determine the Sample value based on Sample_Name
            sample = 'Blank' if 'Blank' in sample_name else 'Sample'

            # Open and read all lines of the file
            with open(file_path, 'r') as file:
                lines = file.readlines()

            # -------------------
            # Extract TIC Data
            # -------------------
            TIC_intensities = []
            in_TIC_section = False
            for i, line in enumerate(lines):
                if 'id: TIC' in line:
                    print("Found 'id: TIC' section.")  # Debug: Found TIC section
                    in_TIC_section = True
                    continue  # Move to next line

                if in_TIC_section:
                    # Look for 'cvParam: intensity array, number of detector counts'
                    if 'cvParam: intensity array, number of detector counts' in line:
                        print("Found 'cvParam: intensity array, number of detector counts'.")  # Debug
                        # The intensity values are in the next line
                        if i + 1 < len(lines):
                            match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', lines[i + 1])
                            if match:
                                TIC_intensities = list(map(int, match.group(1).split()))
                                print(f"Extracted TIC intensities: {TIC_intensities}")  # Debug: TIC values
                                break
                            else:
                                print(f"Warning: No intensity data found after line {i + 1} in file {file_name}")
                        else:
                            print(f"Warning: No lines after 'cvParam: intensity array' in file {file_name}")
                    # If another 'id: ' section starts, stop searching
                    elif re.match(r'id:\s+\w+', line):
                        print("Reached a new 'id:' section before finding TIC intensities.")  # Debug
                        break

            if not TIC_intensities:
                print(f"Warning: No TIC intensities found in file {file_name}")
                TIC_rsd = 0.0  # Default value when TIC intensities are not found
            else:
                # Calculate TIC_RSD using the TIC_RSD function
                TIC_rsd = TIC_RSD(TIC_intensities)
                print(f"Calculated TIC_RSD: {TIC_rsd:.2f}%")  # Debug: TIC RSD value

            # -------------------
            # Parse Lipid Data
            # -------------------
            current_filename = base_name  # Assign filename based on external file name

            current_q1 = None
            current_q3 = None
            current_lipid = ""
            parsing_intensity = False
            intensities = []

            for line in lines:
                # Extract Q1, Q3 values, and Lipid name
                if 'id: SRM SIC Q1=' in line:
                    # Adjusted regex to accommodate additional parameters and quoted names
                    match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name="([^"]+)"', line)
                    if match:
                        current_q1 = float(match.group(1))
                        current_q3 = float(match.group(2))
                        current_lipid = match.group(3)
                        # Removed print statement for Q1 extraction as per user request

                # Check if we are parsing intensity array data for lipids
                if 'cvParam: intensity array' in line and 'number of detector counts' not in line:
                    parsing_intensity = True
                    intensities = []
                    print("Parsing lipid intensity array.")  # Debug
                elif parsing_intensity and 'binary: [' in line:
                    # Extract intensity values
                    match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                    if match:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)
                        print(f"Extracted lipid intensities: {intensities}")  # Debug: Lipid intensity values
                        print(f"Summed Intensity: {current_intensity_sum}")  # Debug: Summed intensity

                        # Append the extracted and calculated data to the lists
                        if current_filename and current_q1 is not None and current_q3 is not None:
                            filenames.append(current_filename)
                            q1_values.append(current_q1)
                            q3_values.append(current_q3)
                            lipids.append(current_lipid)
                            dates.append(date_str)              # Append Date
                            sample_names.append(sample_name)    # Append Sample_Name
                            samples.append(sample)              # Append Sample
                            summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                            tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD
                            print(f"Appended data for lipid: {current_lipid}")  # Debug: Data appended

                    parsing_intensity = False

    # Create a DataFrame
            chromatogram_df = pd.DataFrame({
                'Date': dates,                        # Date column
                'Sample_Name': sample_names,          # Sample_Name column
                'Sample': samples,                    # Sample column
                'Lipid': lipids,
                'Q1': q1_values,
                'Q3': q3_values,
                'Summed_Intensity': summed_intensities,  # Summed_Intensity column
                'TIC_RSD': tic_rsd_values,            # TIC_RSD column
                'Filename': filenames,
            })

            # Ensure that 'Sample_Name' contains only strings
            chromatogram_df['Sample_Name'] = chromatogram_df['Sample_Name'].astype(str)

            # Create Base_Sample_Name by removing 'Blank_' prefix if present
            chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

            # Assign group numbers based on Base_Sample_Name
            chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

            # Optionally, drop the Base_Sample_Name column if not needed
            chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

            # Optionally, convert Date to datetime format
            # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

            # Reorder columns for better readability
            columns_order = [
                'Date', 
                'Sample_Name', 
                'Sample', 
                'Blank_Group', 
                'Lipid', 
                'Q1', 
                'Q3', 
                'Summed_Intensity',
                'TIC_RSD',                           # Include TIC_RSD in the order
                'Filename'
            ]
            chromatogram_df = chromatogram_df[columns_order]

            # Check if all columns have the same length
            lengths = [len(chromatogram_df[col]) for col in chromatogram_df.columns]
            if len(set(lengths)) != 1:
                print("Error: Mismatched list lengths detected. Please check the parsing logic.")
            else:
                print(f"\nAll columns have consistent lengths: {lengths[0]} entries each.")

            # Save the DataFrame to a CSV file
            chromatogram_df.to_csv(output_file, index=False)
            print(f"\nParsing complete. Data saved to {output_file}")  # Debug: Completion message

            return chromatogram_df

# Example usage:
# input_directory = '/path/to/input_directory'
# output_directory = '/path/to/output_directory'
# chromatogram_df = parse_chromatogram_data(input_directory, output_directory)
# print(chromatogram_df.head())


In [18]:
import re
import os
import pandas as pd

def TIC_RSD(intensities):
    """
    Calculate the Relative Standard Deviation (RSD) for a given intensity array.

    Args:
        intensities (list of int): The intensity values.

    Returns:
        float: The RSD percentage.
    """
    if not intensities:
        return 0.0
    mean = sum(intensities) / len(intensities)
    if mean == 0:
        return 0.0
    variance = sum((x - mean) ** 2 for x in intensities) / len(intensities)
    std_dev = variance ** 0.5
    rsd = (std_dev / mean) * 100
    return rsd

def parse_chromatogram_data(input_dir, output_dir):
    """
    Parse chromatogram data from text files in the specified input directory
    and save the results as a CSV in the specified output directory.

    Args:
        input_dir (str): Directory containing the input text files.
        output_dir (str): Directory to save the output CSV file.

    Returns:
        pd.DataFrame: DataFrame containing the parsed chromatogram data.
    """
    # Ensure the output directory exists
    os.makedirs(output_dir, exist_ok=True)
    output_file = os.path.join(output_dir, 'parsed_chromatogram_data.csv')

    # Initialize lists to store parsed data
    filenames = []
    q1_values = []
    q3_values = []
    lipids = []
    dates = []          # List for Date
    sample_names = []   # List for Sample_Name
    samples = []        # List for Sample
    summed_intensities = []  # List for Summed_Intensity
    tic_rsd_values = [] # List for TIC_RSD

    # Iterate through all .txt files in the specified directory
    for file_name in os.listdir(input_dir):
        if file_name.endswith('.txt'):
            file_path = os.path.join(input_dir, file_name)

            # Extract Date and Sample_Name from the filename
            base_name = os.path.splitext(file_name)[0]  # Removes the .txt extension
            parts = base_name.split('_', 1)             # Split only on the first underscore
            if len(parts) == 2:
                date_str = parts[0]                     # e.g., '20241115'
                sample_name = parts[1]                  # e.g., 'Plasma_Acyl-Carnitines'
            else:
                # Handle unexpected filename formats
                date_str = ''
                sample_name = ''

            # Determine the Sample value based on Sample_Name
            sample = 'Blank' if 'Blank' in sample_name else 'Sample'

            # Open and read all lines of the file
            with open(file_path, 'r') as file:
                lines = file.readlines()

            # -------------------
            # Extract TIC Data
            # -------------------
            TIC_intensities = []
            for i, line in enumerate(lines):
                if 'id: TIC' in line:
                    # Look for the intensity array within TIC section
                    for j in range(i, len(lines)):
                        if 'binaryDataArray:' in lines[j] and 'intensity array' in lines[j]:
                            # The intensity values are in the next line
                            if j + 1 < len(lines):
                                match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', lines[j + 1])
                                if match:
                                    TIC_intensities = list(map(int, match.group(1).split()))
                                    break
                    break  # Assuming only one TIC section per file

            # Calculate TIC_RSD using the TIC_RSD function
            TIC_rsd = TIC_RSD(TIC_intensities)

            # -------------------
            # Parse Lipid Data
            # -------------------
            current_filename = ""
            current_q1 = None
            current_q3 = None
            current_lipid = ""
            parsing_intensity = False
            intensities = []

            for line in lines:
                # Extract the filename (assumes filename appears earlier in the file)
                if 'sourceFile:' in line or 'name:' in line:
                    match = re.search(r'name:\s+([\w.]+)', line)
                    if match:
                        current_filename = match.group(1)

                # Extract Q1, Q3 values, and Lipid name
                if 'id: SRM SIC Q1=' in line:
                    match = re.search(r'Q1=(\d+\.\d+).*Q3=(\d+\.\d+).*name=([^\s]+)', line)
                    if match:
                        current_q1 = float(match.group(1))
                        current_q3 = float(match.group(2))
                        current_lipid = match.group(3)

                # Check if we are parsing intensity array data for lipids
                if 'cvParam: intensity array' in line:
                    parsing_intensity = True
                    intensities = []
                elif parsing_intensity and 'binary: [' in line:
                    # Extract intensity values
                    match = re.search(r'binary:\s+\[\d+\]\s+([\d\s]+)', line)
                    if match:
                        intensities = list(map(int, match.group(1).split()))
                        current_intensity_sum = sum(intensities)

                        # Append the extracted and calculated data to the lists
                        if current_filename and current_q1 is not None and current_q3 is not None:
                            filenames.append(current_filename)
                            q1_values.append(current_q1)
                            q3_values.append(current_q3)
                            lipids.append(current_lipid)
                            dates.append(date_str)              # Append Date
                            sample_names.append(sample_name)    # Append Sample_Name
                            samples.append(sample)              # Append Sample
                            summed_intensities.append(current_intensity_sum)  # Append Summed_Intensity
                            tic_rsd_values.append(TIC_rsd)      # Append TIC_RSD

                    parsing_intensity = False

    # Create a DataFrame
    chromatogram_df = pd.DataFrame({
        'Date': dates,                        # Date column
        'Sample_Name': sample_names,          # Sample_Name column
        'Sample': samples,                    # Sample column
        'Lipid': lipids,
        'Q1': q1_values,
        'Q3': q3_values,
        'Summed_Intensity': summed_intensities,  # Summed_Intensity column
        'TIC_RSD': tic_rsd_values,            # TIC_RSD column
        'Filename': filenames,
    })

    # Create Base_Sample_Name by removing 'Blank_' prefix if present
    chromatogram_df['Base_Sample_Name'] = chromatogram_df['Sample_Name'].str.replace('Blank_', '', regex=False)

    # Assign group numbers based on Base_Sample_Name
    chromatogram_df['Blank_Group'] = pd.factorize(chromatogram_df['Base_Sample_Name'])[0] + 1  # Start groups at 1

    # Optionally, drop the Base_Sample_Name column if not needed
    chromatogram_df.drop(columns=['Base_Sample_Name'], inplace=True)

    # Optionally, convert Date to datetime format
    # chromatogram_df['Date'] = pd.to_datetime(chromatogram_df['Date'], format='%Y%m%d')

    # Reorder columns for better readability
    columns_order = [
        'Date', 
        'Sample_Name', 
        'Sample', 
        'Blank_Group', 
        'Lipid', 
        'Q1', 
        'Q3', 
        'Summed_Intensity',
        'TIC_RSD',                           # Include TIC_RSD in the order
        'Filename'
    ]
    chromatogram_df = chromatogram_df[columns_order]

    # Save the DataFrame to a CSV file
    chromatogram_df.to_csv(output_file, index=False)

    return chromatogram_df


In [19]:
# # Import the function from QTRAP.py
# from QTRAP import parse_chromatogram_data

# Define input and output directories
input_dir = 'data/text/'
output_dir = 'result/'

# Parse the chromatogram data and save to a CSV
chromatogram_df = parse_chromatogram_data(input_dir, output_dir)

# Display the resulting DataFrame
chromatogram_df


,Date,Sample_Name,Sample,Blank_Group,Lipid,Q1,Q3,Summed_Intensity,TIC_RSD,Filename
0,20241115,Plasma_TG_22-6,Sample,1,"""[TG(48:7),TG(47:0)]_FA22:6""",810.76,465.46,2440,0.0,20241115_Plasma_TG_22
1,20241115,Plasma_TG_22-6,Sample,1,"""[TG(49:7),TG(48:0)]_FA22:6""",824.77,479.47,1560,0.0,20241115_Plasma_TG_22
2,20241115,Plasma_TG_22-6,Sample,1,"""[TG(50:7),TG(49:0)]_FA22:6""",838.79,493.49,1400,0.0,20241115_Plasma_TG_22
3,20241115,Plasma_TG_22-6,Sample,1,"""[TG(51:7),TG(50:0)]_FA22:6""",852.80,507.50,1560,0.0,20241115_Plasma_TG_22
4,20241115,Plasma_TG_22-6,Sample,1,"""[TG(52:7),TG(51:0)]_FA22:6""",866.82,521.52,1080,0.0,20241115_Plasma_TG_22
...,...,...,...,...,...,...,...,...,...,...
227,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(54:11),TG(53:4)]_FA22:6""",886.79,541.49,800,0.0,20241115_Plasma_Blank_TG_22
228,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(55:10),TG(54:3)]_FA22:6""",902.82,557.52,920,0.0,20241115_Plasma_Blank_TG_22
229,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(55:11),TG(54:4)]_FA22:6""",900.80,555.50,360,0.0,20241115_Plasma_Blank_TG_22
230,20241115,Plasma_Blank_TG_22-6,Blank,1,"""[TG(56:9),TG(55:2)]_FA22:6""",918.85,573.55,1160,0.0,20241115_Plasma_Blank_TG_22


In [14]:
#print unique Sample_Name values
print(chromatogram_df['Sample_Name'].unique())
#sort by Intensity
chromatogram_df.sort_values(by='Summed_Intensity', ascending=False)
# chromatogram_df

[]


,Date,Sample_Name,Sample,Blank_Group,Lipid,Q1,Q3,Summed_Intensity,TIC_RSD,Filename


In [3]:
# print uniqu filename
print(chromatogram_df['Filename'].unique())

['20241115_Plasma_TG_18' '20241115_Plasma_Blank_PI.wiff.scan'
 '20241115_Plasma_FFA' '20241115_Plasma_DG_18' '20241115_Plasma_DG_16'
 '20241115_Plasma_TG_22' '20241115_Plasma_Blank_Acyl'
 '20241115_Plasma_Blank_TG_14' '20241115_Plasma_Blank_DG_16'
 '20241115_Plasma_Blank_TG_18' '20241115_Plasma_Blank_TG_22'
 '20241115_Plasma_TG_16' '20241115_Plasma_Blank_Cholesteryrl'
 '20241115_Plasma_PS.wiff.scan' '20241115_Plasma_Blank_PG.wiff.scan'
 '20241115_Plasma_PG.wiff.scan' '20241115_Plasma_Ceramides.wiff.scan'
 '20241115_Plasma_TG_20' '20241115_Plasma_SM.wiff.scan'
 '20241115_Plasma_Blank_PS.wiff.scan' '20241115_Plasma_PI.wiff.scan'
 '20241115_Plasma_Blank_DB_18' '20241115_Plasma_Cholesteryrl'
 '20241115_Plasma_Acyl' '20241115_Plasma_Blank_TG_20'
 '20241115_Plasma_Blank_TG_16' '20241115_Plasma_PE.wiff.scan'
 '20241115_Plasma_Blank_DG_18' '20241115_Plasma_Blank_FFA'
 '20241115_Plasma_Blank_PC.wiff.scan' '20241115_Plasma_TG_14'
 '20241115_Plasma_Blank_PE.wiff.scan'
 '20241115_Plasma_Blank_Cera

# 2 - QTRAP Plot

In [4]:
from QTRAP_plot import plot_lipid_intensities

# Define input CSV and output directory for plots
input_csv = 'result/parsed_chromatogram_data.csv'
output_dir = 'result/plots/'

# # Generate bar plots sorted by Lipid
# plot_lipid_intensities(input_csv, output_dir, sorting='Lipid')

# Generate bar plots sorted by Intensity
plot_lipid_intensities(input_csv, output_dir, sorting='Intensity')


Processing files:   0%|          | 0/34 [00:00<?, ?it/s]

Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_6_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_7_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_8_lipid_intensities.png


Processing files:   3%|▎         | 1/34 [00:02<01:06,  2.01s/it]

Plot saved: result/plots/20241115_Plasma_Acyl/20241115_Plasma_Acyl_chunk_9_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_6_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_7_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_Blank_Acyl/20241115_Plasma_Blank_Acyl_chunk_8_lipid_intensities.png


Processing files:   6%|▌         | 2/34 [00:02<00:43,  1.36s/it]

...

Plot saved: result/plots/20241115_Plasma_TG_18/20241115_Plasma_TG_18_chunk_65_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_18/20241115_Plasma_TG_18_chunk_66_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_6_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_7_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_8_lipid_intensities.png
Plot sav

Processing files:  97%|█████████▋| 33/34 [01:09<00:03,  3.14s/it]

Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_13_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_20/20241115_Plasma_TG_20_chunk_14_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_1_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_2_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_3_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_4_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_5_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_6_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_7_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_8_lipid_intensities.png
Plot sav

Processing files: 100%|██████████| 34/34 [01:12<00:00,  2.13s/it]

Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_23_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_24_lipid_intensities.png
Plot saved: result/plots/20241115_Plasma_TG_22/20241115_Plasma_TG_22_chunk_25_lipid_intensities.png


['20241023_PlasmaExtract11.wiff.scan' '20241023_Blank31.wiff.scan'
 '20241023_PlasmaExtract13.wiff.scan' '20241023_PlasmaExtract8.wiff.scan'
 '20241023_PlasmaExtract14.wiff.scan' '20241023_PlasmaExtract4.wiff.scan'
 '20241023_PlasmaExtract3.wiff.scan' '20241023_PlasmaExtract12.wiff.scan'
 '20241023_PlasmaExtract21.wiff.scan' '20241023_PlasmaExtract24.wiff.scan'
 '20241023_PlasmaExtract16.wiff.scan' '20241023_PlasmaExtract20.wiff.scan'
 '20241023_PlasmaExtract17.wiff.scan' '20241023_PlasmaExtract23.wiff.scan'
 '20241023_PlasmaExtract15.wiff.scan' '20241023_PlasmaExtract10.wiff.scan'
 '20241023_PlasmaExtract25.wiff.scan' '20241023_PlasmaExtract19.wiff.scan'
 '20241023_PlasmaExtract5.wiff.scan' '20241023_PlasmaExtract18.wiff.scan'
 '20241023_Blank34.wiff.scan' '20241023_PlasmaExtract2.wiff.scan'
 '20241023_Blank33.wiff.scan' '20241023_PlasmaExtract9.wiff.scan'
 '20241023_Blank32.wiff.scan' '20241023_PlasmaExtract1.wiff.scan'
 '20241023_Blank35.wiff.scan' '20241023_PlasmaExtract7.wiff.scan